# Detecção de outliers com Isolation Forest (versão simples)

- Name: Renan Santos Mendes
- Email: renansantosmendes@gmail.com
- Course subject: `<COURSE_SUBJECT_PLACEHOLDER>`
- Program: `<PROGRAM_PLACEHOLDER>`

Versão didática do treino: o código roda de cima para baixo, sem funções.
O modelo é um Isolation Forest que aprende o padrão de retornos, volume e
retorno relativo ao Ibovespa, e marca os dias que fogem desse padrão. A
avaliação é walk-forward: o modelo é reajustado com dados do passado e só
pontua os dias seguintes, que ele ainda não viu.

## 1. Bibliotecas

Importa as bibliotecas usadas no notebook. Para instalar as dependências,
rode `uv sync --extra dev --extra notebook` na raiz do repositório.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.ensemble import IsolationForest

## 2. Parâmetros

Define o ativo, o benchmark e as configurações do walk-forward. Para uma
ação da B3, use o sufixo `.SA` (por exemplo `PETR4.SA`). `CONTAMINATION` é a
fração esperada de outliers em cada treino. O primeiro treino usa cerca de
3 anos de pregões (`TRAIN_MIN`), e o modelo é reajustado a cada cerca de 6
meses (`STEP`).

In [ ]:
TICKER = "AAPL"
BENCH = "^BVSP"
START = "2020-01-01"
CONTAMINATION = 0.1
TRAIN_MIN = 252 * 3
STEP = 126
SEED = 42

## 3. Preços

Baixa o histórico do ativo com preços ajustados e remove o fuso horário do
índice de datas. Ficam só as colunas de fechamento e de volume.

In [ ]:
px = yf.Ticker(TICKER).history(start=START, auto_adjust=True)
px.index = px.index.tz_localize(None)
df = px[["Close", "Volume"]].copy()

## 4. Features de retorno e volume

Cria as features com dados até o próprio dia ou anteriores, para o modelo
não usar informação do futuro:

- **ret**: retorno logarítmico diário.
- **ret_abs**: tamanho do movimento, sem o sinal.
- **ret_lag1**: retorno do dia anterior.
- **vol_20**: volatilidade dos últimos 20 dias.
- **vol_rel**: volume do dia dividido pela média dos últimos 20 dias.

In [ ]:
df["ret"] = np.log(df["Close"]).diff()
df["ret_abs"] = df["ret"].abs()
df["ret_lag1"] = df["ret"].shift(1)
df["vol_20"] = df["ret"].rolling(20).std()
df["vol_rel"] = df["Volume"] / df["Volume"].rolling(20).mean()

## 5. Retorno anormal em relação ao Ibovespa

Calcula `ret_ab`, o retorno do ativo menos o retorno do Ibovespa no mesmo
dia. Se o benchmark não estiver disponível, a coluna fica vazia e o notebook
segue sem ela.

In [ ]:
try:
    bm = yf.download(BENCH, start=START, auto_adjust=True, progress=False)["Close"].squeeze()
    bm.index = pd.DatetimeIndex(bm.index).tz_localize(None)
    df["ret_ab"] = df["ret"] - np.log(bm).diff().reindex(df.index)
except Exception as e:
    print("Benchmark indisponível:", e)
    df["ret_ab"] = np.nan

## 6. Datas de resultados trimestrais

Marca com `is_earn = 1` os dias de divulgação de resultado da empresa. Se
as datas não estiverem disponíveis, a coluna recebe zero em todos os dias.

In [ ]:
try:
    earn = yf.Ticker(TICKER).get_earnings_dates(limit=60)
    earn_idx = pd.DatetimeIndex(earn.index).tz_localize(None).normalize()
    df["is_earn"] = df.index.normalize().isin(earn_idx).astype(int)
except Exception as e:
    print("Datas de resultado indisponíveis:", e)
    df["is_earn"] = 0

## 7. Matriz de features

Escolhe as features que têm dados, troca infinitos por vazio e descarta os
dias incompletos. O modelo só recebe as colunas de `cols`.

In [ ]:
cand = ["ret", "ret_abs", "ret_lag1", "vol_20", "vol_rel", "ret_ab", "is_earn"]
cols = [c for c in cand if df[c].notna().any()]
feat = df[cols].replace([np.inf, -np.inf], np.nan).dropna()
print(f"Dias com features completas: {len(feat)}")

## 8. Walk-forward

Para cada bloco de `STEP` dias, o modelo é treinado só com as linhas
anteriores ao bloco (`train`) e pontua o bloco seguinte (`test`), que ele
ainda não viu. O score é `-score_samples`, então valores maiores indicam
dias mais anômalos. O rótulo `-1` marca outlier.

In [ ]:
if len(feat) <= TRAIN_MIN:
    raise ValueError("Histórico insuficiente para o TRAIN_MIN definido.")

scores = pd.Series(np.nan, index=feat.index)
labels = pd.Series(np.nan, index=feat.index)

for start in range(TRAIN_MIN, len(feat), STEP):
    train = feat[cols].iloc[:start]
    test = feat[cols].iloc[start:start + STEP]

    m = IsolationForest(n_estimators=300, contamination=CONTAMINATION, random_state=SEED)
    m.fit(train)

    scores.iloc[start:start + STEP] = -m.score_samples(test)
    labels.iloc[start:start + STEP] = m.predict(test)

feat["score"] = scores
feat["label"] = labels
df = df.join(feat[["score", "label"]])

## 9. Resultados

Mostra quantos dias foram pontuados fora da amostra de treino, quantos
foram marcados como outlier e os 15 outliers com maior score.

In [ ]:
scored = df[df["label"].notna()]
out = scored[scored["label"] == -1]

print(f"Dias pontuados (out-of-sample): {len(scored)}")
print(f"Outliers detectados: {len(out)} ({len(out) / len(scored):.2%})")
print("\nTop 15 outliers por score:")
print(out.sort_values("score", ascending=False)[["Close", "ret", "vol_rel", "score"]].head(15))

## 10. Gráfico

Desenha o preço com os outliers em vermelho, e embaixo o score de anomalia.
A linha tracejada mostra onde começa o período avaliado pelo walk-forward.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(df.index, df["Close"], lw=1, color="steelblue", label="Fechamento ajustado")
ax1.scatter(out.index, out["Close"], color="red", s=35, zorder=3,
            label=f"Outliers walk-forward ({len(out)})")
ax1.axvline(feat.index[TRAIN_MIN], color="gray", ls="--", lw=1,
            label="Início do período avaliado")
ax1.set_title(f"{TICKER} - outliers com Isolation Forest (walk-forward)")
ax1.set_ylabel("Preço")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(scored.index, scored["score"], lw=0.8, color="gray", label="Score de anomalia")
ax2.scatter(out.index, out["score"], color="red", s=20, zorder=3)
ax2.set_ylabel("Score")
ax2.set_xlabel("Data")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()